In [ ]:
import sys
from setup_paths import paths
sys.path.append(paths["project_dir"])

In [ ]:
import json

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
import scipy.integrate as integrate

from ast import literal_eval
from copy import deepcopy
from matplotlib.colors import rgb2hex, LogNorm, Normalize, CenteredNorm, LinearSegmentedColormap
from matplotlib_scalebar.scalebar import ScaleBar
from mpl_toolkits.axes_grid1 import make_axes_locatable
from os.path import join
from pathlib import Path
from scipy import stats
from shapely.geometry import Point

from mode_choice import get_od_stations
from plot_osmnx import plot_used_nodes, plot_graph
from src.prep_and_plot.data_helper import load_demand, get_polygon_from_json

In [ ]:
def truncate_colormap(old_cmap, minval=0.0, maxval=1.0, n=100):
    new_cmap = LinearSegmentedColormap.from_list(
        'trunc({n},{a:.2f},{b:.2f})'.format(n=old_cmap.name, a=minval, b=maxval),
        old_cmap(np.linspace(minval, maxval, n)))
    return new_cmap

In [ ]:
save = "dd"
global_crs = 4326
local_crs = 32633

modes_dict = {
    "CyclistsLengthNormed": "000",
    "NormalizedTravelTimePenalty": "010",
    "TotalTravelTimeEstimation": "020",
    "TotalTravelTimeEstimationPerLength": "030",
    "TotalTravelTimeEstimationCapped": "040",
    "TotalTravelTimeEstimationCappedPerLength": "050",
    "RandomOrder": "080",
}
modes = list(modes_dict.values())
modes.extend([f"{m[:2]}1" for m in modes])
mode_base = modes_dict["TotalTravelTimeEstimationPerLength"]
mode_base_exinf = f"{mode_base[:2]}1"
modes_dict_rev = {k: {v_i[1]: k_i for k_i, v_i in modes_dict.items()}[k[1]] for k in modes}

labels = {
    "CyclistsLengthNormed": r"$Q_\mathrm{trips}$",
    "NormalizedTravelTimePenalty": r"$Q_\mathrm{pen}$",
    "TotalTravelTimeEstimation": r"$Q_{\widetilde{\Delta T}}$",
    "TotalTravelTimeEstimationPerLength": r"$q_{\widetilde{\Delta T}}$",
    "TotalTravelTimeEstimationCapped": r"$Q_{\widetilde{\Delta T}, \mathrm{capped}}$",
    "TotalTravelTimeEstimationCappedPerLength": r"$q_{\widetilde{\Delta T}, \mathrm{capped}}$",
    "TotalDetour": r"$Q_{\Delta T}$",
    "TotalDetourPerLength": r"$q_{\Delta T}$",
    "TotalTravelTimeEstimationUncapped": r"$Q_{\widetilde{\Delta T}, \mathrm{uncapped}}$",
    "RandomOrder": "random",
}
colors = {
    "CyclistsLengthNormed": "#4363d8",
    "NormalizedTravelTimePenalty": "#000075",
    "TotalTravelTimeEstimation": "#800000", #dcbeff
    "TotalTravelTimeEstimationPerLength": "#800000",
    "TotalTravelTimeEstimationCapped": "#ffe119", #f58231
    "TotalTravelTimeEstimationCappedPerLength": "#ffe119", #800000
    "TotalDetour": "#ffe119", #ffc819
    "TotalDetourPerLength": "#ffe119",
    "TotalTravelTimeEstimationUncapped": "#a9a9a9",
    "RandomOrder": "#000000",
}
line_style = {
    "CyclistsLengthNormed": "--",
    "NormalizedTravelTimePenalty": "--",
    "TotalTravelTimeEstimation": "--",
    "TotalTravelTimeEstimationPerLength": "-",
    "TotalTravelTimeEstimationCapped": "--",
    "TotalTravelTimeEstimationCappedPerLength": "-",
    "TotalDetour": "--",
    "TotalDetourPerLength": "-",
    "TotalTravelTimeEstimationUncapped": "--",
    "RandomOrder": "--",
}

color_street_nw_buffered = "#999999"
color_street_nw  = "#4d4d4d"
color_street_nw_exinf = "k"
color_street_nw_both = "#f58231" #bb4219

outlier_threshold = 1200
divnorm_ranking = Normalize(vmin=-outlier_threshold, vmax=outlier_threshold, clip=True)

street_type_colors = {"primary": "C0", "secondary": "C1", "tertiary": "C2", "residential": "C5", "bike_path": "C4"}
random_realisations = 10

modes_noisy = ["000", "001", "010", "011", "030", "031", "050", "051"]
noise_params = ["noisy_demand", "noisy_car_penalty", "noisy_slope_penalty", "noisy_intersection_penalty", "noisy_turn_penalty"]
noise_samples = 10
noise_labels = {
    "noisy_demand": "Noisy demand",
    "noisy_car_penalty": "Noisy car penalty",
    "noisy_slope_penalty": "Noisy slope penalty",
    "noisy_intersection_penalty": "Noisy intersection penalty",
    "noisy_turn_penalty": "Noisy turn penalty",
}

plot_params = {"dpi": 300, "plot_format": "svg", "figsize": (2.8, 2.2), "figsize_wide": (5.4, 2.2), "figsize_map": (3, 3), "figs_snetwork": (5, 5), "fs_title": 10, "fs_ax_label": 6, "fs_ax_ticks": 6, "fs_cbar_label": 6, "fs_cbar_ticks": 6, "edge_lw_thick": 0.75, "edge_lw_thin": 0.25, "edge_color": "#808080", "node_size": 0.25}

cmap_demand = mpl.colormaps["Greys"]

In [ ]:
polygon = get_polygon_from_json(join(paths["data_dir"], "polygons", f"{save}.json"))
polygon_projected = gpd.GeoDataFrame({"geometry": [polygon]}, geometry="geometry", crs=global_crs)
polygon_projected = polygon_projected.to_crs(local_crs)["geometry"][0]

In [ ]:
g_buffered = ox.load_graphml(join(paths["input_folder"], save, f"{save}_buffered.graphml"))
for u, v, d in g_buffered.edges(data=True):
    d["ex_inf"] = literal_eval(d["ex_inf"])
g = ox.load_graphml(join(paths["input_folder"], save, f"{save}.graphml"))
for u, v, d in g.edges(data=True):
    d["ex_inf"] = literal_eval(d["ex_inf"])

edge2segid = {(u, v): idx for idx, (u, v) in enumerate(g.edges(), start=1)}
segid2edge = {idx: e for e, idx in edge2segid.items()}

In [ ]:
total_nw_len = sum(d for u, v, d in g_buffered.edges(data="length")) / 1000
total_ex_inf_len = sum(d['length'] for u, v, d in g_buffered.edges(data=True) if d['ex_inf'] or d['highway'] == 'bike_path')  / 1000
print(f"Buffered network, nodes: {nx.number_of_nodes(g_buffered)}, edges: {nx.number_of_edges(g_buffered)}.")
print(f"Buffered network, total network length: {total_nw_len:.0f} km")
print(f"Buffered network, existing infrastructure length: {total_ex_inf_len:.0f} km, relative {total_ex_inf_len/total_nw_len:.3f}")

In [ ]:
total_nw_len = sum(d for u, v, d in g.edges(data="length")) / 1000
total_ex_inf_len = sum(d['length'] for u, v, d in g.edges(data=True) if d['ex_inf'] or d['highway'] == 'bike_path')  / 1000
print(f"Observed network, nodes: {nx.number_of_nodes(g)}, edges: {nx.number_of_edges(g)}.")
print(f"Observed network length: {total_nw_len:.0f} km")
print(f"Observed network existing infrastructure length: {total_ex_inf_len:.0f} km, relative {total_ex_inf_len/total_nw_len:.3f}")

In [ ]:
edge_grade = [d for _, _, d in g.edges(data="grade")]
edge_color = list()
edge_zorder = list()
for d in edge_grade:
    if d > 0.06:
        edge_color.append("r")
        edge_zorder.append(4)
    elif d > 0.04:
        edge_color.append("C1")
        edge_zorder.append(3)
    elif d > 0.02:
        edge_color.append("y")
        edge_zorder.append(2)
    else:
        edge_color.append("#808080")
        edge_zorder.append(1)
plot_graph(g, figsize=(20, 20), node_size=0, edge_color=edge_color, edge_zorder=edge_zorder, edge_linewidth=0.5)

In [ ]:
trips = load_demand(join(paths["data_dir"], "input", save, f"{save}_demand.json"))
trips_unique = {k: sum(v.values()) for k, v in trips.items()}
stations = {i for od in trips.keys() for i in od}

# Data

In [ ]:
local_pop_dist = gpd.read_file(join(paths["data_dir"], "raw", f"dd_pop_inner_split.gpkg"))
local_pop_dist.to_crs(local_crs, inplace=True)
local_pop_dist["centroid"] = local_pop_dist.centroid

demand = pd.read_csv(join(paths["data_dir"], "csvs", f"{save}.csv"))

g_inside = ox.truncate.truncate_graph_polygon(g_buffered, polygon)
g_outside = g_buffered.copy()
edges_inside = [e for e in g_buffered.edges() if e in g_inside.edges()]
g_outside.remove_edges_from(edges_inside)

gdf_buffered = ox.graph_to_gdfs(g_buffered, nodes=False).to_crs(local_crs)
gdf_inside = ox.graph_to_gdfs(g_inside, nodes=False).to_crs(local_crs)
gdf_outside = ox.graph_to_gdfs(g_outside, nodes=False).to_crs(local_crs)

h_buffered = ox.project_graph(g_buffered, to_crs=local_crs)
nx.set_node_attributes(h_buffered, {n: polygon_projected.contains(Point((d["x"], d["y"]))) for n, d in h_buffered.nodes(data=True)}, name="in_poly")
_, stations_geom, node_mapping_g_buffered = get_od_stations(local_pop_dist, h_buffered)
stations_geom["geometry_boundary"] = stations_geom.set_geometry("geometry").boundary
stations_geom["area"] = stations_geom.set_geometry("geometry").area
stations_geom["pop_density"] = stations_geom["population"] / stations_geom["area"]

In [ ]:
fig_g_buffered, ax_g_buffered = plt.subplots(dpi=plot_params["dpi"], figsize=plot_params["figsize_map"])
fig_g_buffered.tight_layout()

gdf_inside.plot(ax=ax_g_buffered, color=color_street_nw, lw=plot_params["edge_lw_thin"])
gdf_outside.plot(ax=ax_g_buffered, color=color_street_nw_buffered, lw=plot_params["edge_lw_thin"])
ax_g_buffered.add_artist(ScaleBar(dx=1, sep=1, font_properties={"size": plot_params["fs_ax_label"]}, pad=0))

ax_g_buffered.set_axis_off()

fig_g_buffered.savefig(join(paths["plot_folder"], save, f"{save}_buffered_graph.{plot_params['plot_format']}"), bbox_inches="tight")
plt.show()

In [ ]:
fig_g_exinf, ax_g_exinf = plt.subplots(dpi=plot_params["dpi"], figsize=plot_params["figsize_map"])
fig_g_exinf.tight_layout()

gdf_inside.plot(ax=ax_g_exinf, color=[color_street_nw_exinf if e_exinf else color_street_nw for e_exinf in gdf_inside["ex_inf"]], lw=[2*plot_params["edge_lw_thin"] if e_exinf else plot_params["edge_lw_thin"] for e_exinf in gdf_inside["ex_inf"]])
ax_g_exinf.add_artist(ScaleBar(dx=1, sep=1, font_properties={"size": plot_params["fs_ax_label"]}, pad=0))

ax_g_exinf.set_axis_off()

fig_g_exinf.savefig(join(paths["plot_folder"], save, f"{save}_graph_exinf.{plot_params['plot_format']}"), bbox_inches="tight")
plt.show()

In [ ]:
total_nw_len = sum(d for u, v, d in g.edges(data="length")) / 1000
total_ex_inf_len = sum(d['length'] for u, v, d in g.edges(data=True) if d['ex_inf'] or d['highway'] == 'bike_path')  / 1000
total_ex_inf_len_dedicated = sum([d["length"] for _, _, d in g.edges(data=True) if d["ex_inf"] and not d["highway"] == "bike_path"]) / 1000
print(f"Observed network, nodes: {nx.number_of_nodes(g)}, edges: {nx.number_of_edges(g)}.")
print(f"Observed network length: {total_nw_len:.0f} km")
print(f"Observed network existing infrastructure length: {total_ex_inf_len:.0f} km ({total_ex_inf_len_dedicated:.0f} km dedicated), relative {total_ex_inf_len/total_nw_len:.3f}")


In [ ]:
fig_pop, ax_pop = plt.subplots(dpi=plot_params["dpi"], figsize=plot_params["figsize_map"])
fig_pop.tight_layout()

population_var = "pop_density" # pop_density population
stations_geom["pop_plot"] = stations_geom[population_var] * 1000000
gdf_inside.plot(ax=ax_pop, color=color_street_nw, lw=1.5*plot_params["edge_lw_thin"])
gdf_outside.plot(ax=ax_pop, color=color_street_nw_buffered, lw=1.5*plot_params["edge_lw_thin"])

#stations_geom.set_geometry("geometry_boundary").plot(ax=ax_pop, color="k", lw=plot_params["edge_lw_thin"], zorder=10, alpha=0.7)
stations_geom.set_geometry("geometry").plot(ax=ax_pop, column="pop_plot", cmap=cmap_demand, edgecolor="black", linewidth=plot_params["edge_lw_thin"], alpha=0.6, zorder=10)

ax_pop.add_artist(ScaleBar(dx=1, sep=1, font_properties={"size": plot_params["fs_ax_label"]}))
ax_pop.set_axis_off()

plt.show()
fig_pop.savefig(join(paths["plot_folder"], save, f"{save}_pop_dens.{plot_params['plot_format']}"), bbox_inches="tight")

sm = plt.cm.ScalarMappable(
    cmap=cmap_demand,
    norm=Normalize(vmin=stations_geom["pop_plot"].min(), vmax=stations_geom["pop_plot"].max())
)
divider = make_axes_locatable(ax_pop)
cbaxes = divider.append_axes("bottom", size="5%", pad=0.05)
cbar = fig_pop.colorbar(
    sm,
    ax=ax_pop,
    orientation="horizontal",
    cax=cbaxes,
)
cbar.set_label(label="population density [$km^{-2}$]", size=plot_params["fs_cbar_label"])
cbar.ax.tick_params(labelsize=plot_params["fs_cbar_ticks"])

ax_pop.remove()
fig_pop.savefig(join(paths["plot_folder"], save, f"{save}_pop_dens_cbar.{plot_params['plot_format']}"), bbox_inches="tight")

In [ ]:
od_pair_stations = (1388, 1269)
od_pair_nodes = (node_mapping_g_buffered[od_pair_stations[0]], node_mapping_g_buffered[od_pair_stations[1]])
od_path = ox.shortest_path(g_buffered, orig=od_pair_nodes[0], dest=od_pair_nodes[1])
od_path_edges = [(n, od_path[idx+1]) for idx, n in enumerate(od_path[:-1])]

g_path = deepcopy(g_buffered)
g_path.remove_edges_from([e for e in g_buffered.edges() if e not in od_path_edges])
g_path.remove_nodes_from(list(nx.isolates(g_path)))
gdf_path_edges = ox.graph_to_gdfs(g_path, nodes=False).to_crs(local_crs)
gdf_path_nodes = ox.convert.graph_to_gdfs(ox.project_graph(g_path, to_crs=local_crs), edges=False, node_geometry=False)[["x", "y"]]

mapped_nodes = ox.nearest_nodes(
    g_buffered,
    [demand[(demand["start station"] == od_pair_stations[0]) & (demand["end station"] == od_pair_stations[1])]["start longitude"].iloc[0],
    demand[(demand["start station"] == od_pair_stations[0]) & (demand["end station"] == od_pair_stations[1])][ "end longitude"].iloc[0]],
    [demand[(demand["start station"] == od_pair_stations[0]) & (demand["end station"] == od_pair_stations[1])]["start latitude"].iloc[0],
     demand[(demand["start station"] == od_pair_stations[0]) & (demand["end station"] == od_pair_stations[1])]["end latitude"].iloc[0]]
)

In [ ]:
fig_path, ax_path = plt.subplots(dpi=plot_params["dpi"], figsize=plot_params["figsize_map"])
fig_path.tight_layout()

gdf_inside.plot(ax=ax_path, color=color_street_nw, lw=plot_params["edge_lw_thin"])
gdf_outside.plot(ax=ax_path, color=color_street_nw_buffered, lw=plot_params["edge_lw_thin"])

gdf_path_edges.plot(ax=ax_path, color="k", lw=1.5*plot_params["edge_lw_thick"], zorder=5)
ax_path.scatter(x=gdf_path_nodes["x"], y=gdf_path_nodes["y"], s=[25*plot_params["node_size"] if n in mapped_nodes else 0 for n in g_path.nodes()], c="k", zorder=10)

ax_path.add_artist(ScaleBar(dx=1, sep=1, font_properties={"size": plot_params["fs_ax_label"]}))
ax_path.set_axis_off()

fig_path.savefig(join(paths["plot_folder"], save, f"{save}_path.{plot_params['plot_format']}"), bbox_inches="tight")
plt.show()

# Results

In [ ]:
order_segment = pd.DataFrame()
utility = pd.DataFrame()
bpp = pd.DataFrame()

for mode in [m for m in modes if m[1] != "8"]:
    print(f"Loading data for mode {mode}.")

    with open(join(paths["output_folder"], save, f"{save}_order_{mode}.json"), "r") as f:
        order_edges_mode = json.load(f)
    order_segment[mode] = [edge2segid[tuple(e)] for e in reversed(order_edges_mode)]

    with open(join(paths["output_folder"], save, f"{save}_bpp_{mode}.json"), "r") as f:
        bpp_mode = json.load(f)
    bpp[mode] = list(reversed(bpp_mode))

    with open(join(paths["output_folder"], save, f"{save}_utility_{mode}.json"), "r") as f:
        utility_raw_mode = json.load(f)
    utility_mode = pd.DataFrame(list(reversed(utility_raw_mode)))
    utility[mode] = utility_mode.replace(to_replace=0.0, value=np.nan).dropna()[0].to_list()

In [ ]:
utility_random = pd.DataFrame()
utility_random_interpol = pd.DataFrame()
bpp_random = pd.DataFrame()

for n in range(1, random_realisations+1):
    with open(join(paths["output_folder"], save, f"{save}_bpp_080_{n}.json"), "r") as f:
        bpp_realisation = json.load(f)
    bpp_random[f"080_{n:02d}"] = list(reversed(bpp_realisation))
    with open(join(paths["output_folder"], save, f"{save}_bpp_081_{n}.json"), "r") as f:
        bpp_realisation = json.load(f)
    bpp_random[f"081_{n:02d}"] = list(reversed(bpp_realisation))

    with open(join(paths["output_folder"], save, f"{save}_utility_080_{n}.json"), "r") as f:
        utility_raw_realisation = json.load(f)
    utility_realisation = pd.DataFrame(list(reversed(utility_raw_realisation)))
    utility_random[f"080_{n:02d}"] = utility_realisation.replace(to_replace=0.0, value=np.nan).dropna()[0].to_list()
    with open(join(paths["output_folder"], save, f"{save}_utility_081_{n}.json"), "r") as f:
        utility_raw_realisation = json.load(f)
    utility_realisation = pd.DataFrame(list(reversed(utility_raw_realisation)))
    utility_random[f"081_{n:02d}"] = utility_realisation.replace(to_replace=0.0, value=np.nan).dropna()[0].to_list()

for n in range(1, random_realisations+1):
    utility_random_interpol[f"080_{n:02d}"] = np.interp(bpp_random["080_01"], bpp_random[f"080_{n:02d}"], utility_random[f"080_{n:02d}"])
    utility_random_interpol[f"081_{n:02d}"] = np.interp(bpp_random["081_01"], bpp_random[f"081_{n:02d}"], utility_random[f"081_{n:02d}"])

bpp["080"] = bpp_random["080_01"]
utility["080"] = utility_random_interpol[[f"080_{n:02d}" for n in range(1, random_realisations+1)]].mean(axis=1)
bpp["081"] = bpp_random["081_01"]
utility["081"] = utility_random_interpol[[f"081_{n:02d}" for n in range(1, random_realisations+1)]].mean(axis=1)

In [ ]:
ba = pd.DataFrame()
utility_opt = utility.max().max()
utility_base = utility.min().min()
for mode in modes:
    ba[mode] = (utility_base - utility[mode]) / (utility_base - utility_opt)

In [ ]:
ba_diff = pd.DataFrame()
for mode in modes:
    ba_comp_interpol = np.interp(bpp[mode], bpp[mode_base], ba[mode_base])
    ba_diff[mode] = (ba[mode]-ba_comp_interpol)/ba_comp_interpol

In [ ]:
plt.plot(bpp["050"][:600], ba_diff["050"][:600])
plt.show()

In [ ]:
ba_80_bpp = {mode: bpp[mode].loc[ba[ba[mode]>=0.8].first_valid_index()] for mode in modes if mode[1] != "8" and mode[2] != "1"}
ba_80_bpp_min = min(ba_80_bpp.values())
ba_80_bpp_max = max(ba_80_bpp.values())

ba_80_bpp_exinf = {mode: bpp[mode].loc[ba[ba[mode]>=0.8].first_valid_index()] for mode in modes if mode[1] != "8" and mode[2] != "0"}
ba_80_bpp_min_exinf = min(ba_80_bpp_exinf.values())
ba_80_bpp_max_exinf = max(ba_80_bpp_exinf.values())

In [ ]:
ba_100_bpp = {mode: bpp[mode].loc[ba[ba[mode]>=1.0].first_valid_index()] for mode in modes if mode[1] != "8" and mode[2] != "1"}
ba_100_bpp_exinf = {mode: bpp[mode].loc[ba[ba[mode]>=1.0].first_valid_index()] for mode in modes if mode[1] != "8" and mode[2] != "0"}

In [ ]:
print(f"{len(trips_unique.keys())} unique trips, with {sum(trips_unique.values()):1.0f} travellers, between {len(stations)} od nodes")
fig_nw_demand, ax_nw_demand = plot_used_nodes(
    g,
    trips=trips_unique,
    stations=stations,
    cmap=cmap_demand,
    norm=LogNorm,
    plot_params=plot_params,
    extend="neither",
    cbar_label="node usage"
)
#fig_nw_demand.savefig(join(paths["plot_folder"], name_full, f"FIG2b_demand.{save_format}"), bbox_inches="tight")
plt.show()

In [ ]:
fig_ba, ax_ba = plt.subplots(ncols=2, figsize=(6, 2.5), dpi=plot_params["dpi"], sharey=True, sharex=True)
ax_ba_in = ax_ba[0].inset_axes([0.46, 0.07, 0.5, 0.5])
ax_ba_in_exinf = ax_ba[1].inset_axes([0.46, 0.07, 0.5, 0.5])

for mode_label, mode in modes_dict.items():
    mode_exinf = f"{mode[:2]}1"

    ax_ba[0].plot(bpp[mode], ba[mode], ls=line_style[mode_label], lw=0.75, color=colors[mode_label], label=labels[mode_label])
    ax_ba[1].plot(bpp[mode_exinf], ba[mode_exinf], ls=line_style[mode_label], lw=0.75, color=colors[mode_label], label=labels[mode_label])

    if mode not in [mode_base, "080"]:
        ba_comp_interpol = np.interp(bpp[mode], bpp[mode_base], ba[mode_base])
        ba_comp_interpol_exinf = np.interp(bpp[mode_exinf], bpp[mode_base_exinf], ba[mode_base_exinf])

        ax_ba_in.plot(bpp[mode], (ba[mode]-ba_comp_interpol)/ba_comp_interpol, ls=line_style[mode_label], lw=0.5, color=colors[mode_label], label=labels[mode_label])
        ax_ba_in_exinf.plot(bpp[mode_exinf], (ba[mode_exinf]-ba_comp_interpol_exinf)/ba_comp_interpol_exinf, ls=line_style[mode_label], lw=0.5, color=colors[mode_label], label=labels[mode_label])

ax_ba[0].hlines(0.8, 0, ba_80_bpp_max, ls="--", lw=0.5, color="#808080", zorder=0)
ax_ba[0].vlines(ba_80_bpp_min, 0, 0.8, ls="--", lw=0.5, color="#808080", zorder=0)
ax_ba[0].vlines(ba_80_bpp_max, 0, 0.8, ls="--", lw=0.5, color="#808080", zorder=0)
ax_ba[1].hlines(0.8, 0, ba_80_bpp_max_exinf, ls="--", lw=0.5, color="#808080", zorder=0)
ax_ba[1].vlines(ba_80_bpp_min_exinf, 0, 0.8, ls="--", lw=0.5, color="#808080", zorder=0)
ax_ba[1].vlines(ba_80_bpp_max_exinf, 0, 0.8, ls="--", lw=0.5, color="#808080", zorder=0)

ax_ba[0].set_xlabel("Normalised length of bike paths $\lambda$", fontsize=plot_params["fs_ax_label"])
ax_ba[1].set_xlabel("Normalised length of bike paths $\lambda$", fontsize=plot_params["fs_ax_label"])
ax_ba[0].set_ylabel("Bikeability $B(\lambda)$", fontsize=plot_params["fs_ax_label"])

ax_ba[0].tick_params(axis="x", labelsize=plot_params["fs_ax_ticks"])
ax_ba[1].tick_params(axis="x", labelsize=plot_params["fs_ax_ticks"])
ax_ba[0].tick_params(axis="y", labelsize=plot_params["fs_ax_ticks"])

ax_ba[0].set_aspect("equal")
ax_ba[1].set_aspect("equal")

ax_ba[0].set_xlim(0, 1)
ax_ba[0].set_ylim(0, 1)

ax_ba[0].set_title("W/o existing infrastructure", fontsize=plot_params["fs_ax_label"])
ax_ba[1].set_title("With existing infrastructure", fontsize=plot_params["fs_ax_label"])

ax_ba[1].legend(fontsize=plot_params["fs_ax_ticks"], loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False)

ax_ba_in.axhline(y=0, xmin=0, xmax=1, color="#808080", alpha=0.5, ls="--", lw=0.75)
ax_ba_in_exinf.axhline(y=0, xmin=0, xmax=1, color="#808080", alpha=0.5, ls="--", lw=0.75)

ax_ba_in.set_ylabel("$\Delta B(\lambda)$ [%]", fontsize=plot_params["fs_ax_label"], labelpad=1.5)
ax_ba_in_exinf.set_ylabel("$\Delta B(\lambda)$ [%]", fontsize=plot_params["fs_ax_label"], labelpad=1.5)

ax_ba_in.tick_params(axis="x", direction="in", labelsize=plot_params["fs_ax_ticks"]-2, pad=1.5)
ax_ba_in.tick_params(axis="y", direction="in", labelsize=plot_params["fs_ax_ticks"]-2, pad=1.5)
ax_ba_in_exinf.tick_params(axis="x", direction="in", labelsize=plot_params["fs_ax_ticks"]-2, pad=1.5)
ax_ba_in_exinf.tick_params(axis="y", direction="in", labelsize=plot_params["fs_ax_ticks"]-2, pad=1.5)

ax_ba_in.set_xlim(0, 1)
ax_ba_in.set_ylim(-0.3, 0.15)
ax_ba_in.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0, symbol=None))
ax_ba_in.xaxis.set_major_locator(ticker.MultipleLocator(0.2))
ax_ba_in_exinf.set_xlim(0, 1)
ax_ba_in_exinf.set_ylim(-0.15, 0.07)
ax_ba_in_exinf.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0, symbol=None))
ax_ba_in_exinf.xaxis.set_major_locator(ticker.MultipleLocator(0.2))
ax_ba_in_exinf.yaxis.set_major_locator(ticker.MultipleLocator(0.05))

fig_ba.savefig(join(paths["plot_folder"], save, f"{save}_ba.{plot_params['plot_format']}"), bbox_inches="tight")
plt.show()

In [ ]:
ba_diff_area_random_steps = pd.DataFrame()
ba_diff_area_random_steps_exinf = pd.DataFrame()

for mode_label, mode in modes_dict.items():
    mode_exinf = f"{mode[:2]}1"
    if mode[1] != "8":
        ba_random_interpol = np.interp(bpp[mode], bpp["080"], ba["080"])
        ba_diff_area_random_steps[mode] = [np.trapezoid(ba[mode][:idx] - ba_random_interpol[:idx], bpp[mode][:idx]) for idx in range(bpp.shape[0])]
        ba_random_interpol_exinf = np.interp(bpp[mode_exinf], bpp["081"], ba["081"])
        ba_diff_area_random_steps_exinf[mode_exinf] = [np.trapezoid(ba[mode_exinf][:idx] - ba_random_interpol_exinf[:idx], bpp[mode_exinf][:idx]) for idx in range(bpp.shape[0])]

ba_diff_area_random = pd.DataFrame(data={"mode": [mode_label for mode_label, mode in modes_dict.items() if mode[1] != "8"], "absolute": ba_diff_area_random_steps.iloc[-1], "relative": 100*ba_diff_area_random_steps.iloc[-1]/ba_diff_area_random_steps.iloc[-1].max()})
ba_diff_area_random_exinf = pd.DataFrame(data={"mode": [mode_label for mode_label, mode in modes_dict.items() if mode[1] != "8"], "absolute": ba_diff_area_random_steps_exinf.iloc[-1], "relative": 100*ba_diff_area_random_steps_exinf.iloc[-1]/ba_diff_area_random_steps_exinf.iloc[-1].max()});

In [ ]:
ba_diff_area_random.round({"absolute": 3, "relative": 1})

In [ ]:
ba_diff_area_random_exinf.round({"absolute": 3, "relative": 1})

In [ ]:
(np.array(ba_diff_area_random["absolute"]) - np.array(ba_diff_area_random_exinf["absolute"]))/ba_diff_area_random_exinf["absolute"]

In [ ]:
unused_segments = dict()
used_segments = dict()
for mode in [m for m in modes if m[1] != "8"]:
    mode_cut = ba[mode].loc[(ba[mode]>=1.0)].index[0]
    unused_segments[mode] = order_segment[mode].to_list()[mode_cut:]
    used_segments[mode] = order_segment[mode].to_list()[:mode_cut]
    print(f"{mode}: {mode_cut}")

In [ ]:
ordering = pd.DataFrame({"segment": sorted(order_segment["000"])})

for mode in [m for m in modes if m[1] != "8"]:
    ordering[mode] = ordering["segment"].map({seg: idx for idx, seg in enumerate(order_segment[mode], start=1)})

ordering_diff = pd.DataFrame({"segment": sorted(order_segment["000"])})
for mode in [m for m in modes if m[1] != "8"]:
    if mode[2] == "0":
        ordering_diff[mode] = ordering[mode_base] - ordering[mode]
    elif mode[2] == "1":
        ordering_diff[mode] = ordering[mode_base_exinf] - ordering[mode]

In [ ]:
fig_rank, ax_rank = plt.subplots(ncols=2, figsize=(6, 3), dpi=plot_params["dpi"], sharey=True, sharex=True)

mode_comp = "010"
mode_comp_exinf = f"{mode_comp[:2]}1"

ordering_unused = pd.DataFrame({
    "segment": ordering[ordering["segment"].isin(unused_segments[mode_base]) & ordering["segment"].isin(unused_segments[mode_comp])]["segment"],
    mode_base: ordering[ordering["segment"].isin(unused_segments[mode_base]) & ordering["segment"].isin(unused_segments[mode_comp])][mode_base],
    mode_comp: ordering[ordering["segment"].isin(unused_segments[mode_base]) & ordering["segment"].isin(unused_segments[mode_comp])][mode_comp],
})

ordering_unused_exinf = pd.DataFrame({
    "segment": ordering[ordering["segment"].isin(unused_segments[mode_base_exinf]) & ordering["segment"].isin(unused_segments[mode_comp_exinf])]["segment"],
    mode_base_exinf: ordering[ordering["segment"].isin(unused_segments[mode_base_exinf]) & ordering["segment"].isin(unused_segments[mode_comp_exinf])][mode_base_exinf],
    mode_comp_exinf: ordering[ordering["segment"].isin(unused_segments[mode_base_exinf]) & ordering["segment"].isin(unused_segments[mode_comp_exinf])][mode_comp_exinf]
})

ordering_used = ordering[~ordering["segment"].isin(ordering_unused["segment"])][["segment", mode_base, mode_comp]]

ordering_used_exinf = ordering[~ordering["segment"].isin(ordering_unused_exinf["segment"])][["segment", mode_base_exinf, mode_comp_exinf]]

segment_ordering_color = [street_type_colors[g.edges[segid2edge[seg]+(0,)]["highway"]] for seg in ordering_used["segment"]]
segment_ordering_color_exinf = [street_type_colors[g.edges[segid2edge[seg]+(0,)]["highway"]] for seg in ordering_used_exinf["segment"]]

ax_rank[0].plot(np.arange(0, 16000), np.arange(0, 16000), "--", c="#808080", alpha=0.75, lw=0.5, zorder=2)
ax_rank[0].plot(np.arange(0, 16000), np.arange(0, 16000)+outlier_threshold, "--", c="#808080", alpha=0.5, lw=0.5, zorder=2)
ax_rank[0].plot(np.arange(0, 16000), np.arange(0, 16000)-outlier_threshold, "--", c="#808080", alpha=0.5, lw=0.5, zorder=2)
ax_rank[0].scatter(ordering_unused[mode_base], ordering_unused[mode_comp], s=2, c=color_street_nw, alpha=0.5, linewidths=0, zorder=3)
ax_rank[0].scatter(ordering_used[mode_base], ordering_used[mode_comp], s=2, c=segment_ordering_color, linewidths=0, zorder=3)

ax_rank[1].plot(np.arange(0, 16000), np.arange(0, 16000), "--", c="#808080", alpha=0.75, lw=0.5, zorder=2)
ax_rank[1].plot(np.arange(0, 16000), np.arange(0, 16000)+outlier_threshold, "--", c="#808080", alpha=0.5, lw=0.5, zorder=2)
ax_rank[1].plot(np.arange(0, 16000), np.arange(0, 16000)-outlier_threshold, "--", c="#808080", alpha=0.5, lw=0.5, zorder=2)
ax_rank[1].scatter(ordering_unused_exinf[mode_base_exinf], ordering_unused_exinf[mode_comp_exinf], s=2, c=color_street_nw, alpha=0.5, linewidths=0, zorder=3)
ax_rank[1].scatter(ordering_used_exinf[mode_base_exinf], ordering_used_exinf[mode_comp_exinf], s=2, c=segment_ordering_color_exinf, linewidths=0, zorder=3)

ax_rank[0].set_xlim(0, 16000)
ax_rank[0].set_ylim(0, 16000)
ax_rank[0].set_aspect("equal")
ax_rank[1].set_aspect("equal")

ax_rank[0].tick_params(axis="both", labelsize=plot_params["fs_ax_ticks"])
ax_rank[1].tick_params(axis="both", labelsize=plot_params["fs_ax_ticks"])
ax_rank[0].ticklabel_format(axis="both", style="sci", scilimits=(-4, 4), useMathText=True)
ax_rank[1].ticklabel_format(axis="both", style="sci", scilimits=(-4, 4), useMathText=True)
ax_rank[0].xaxis.set_major_locator(ticker.MultipleLocator(2500))
ax_rank[0].yaxis.set_major_locator(ticker.MultipleLocator(2500))
ax_rank[0].xaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])
ax_rank[1].xaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])
ax_rank[0].yaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])
ax_rank[1].yaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])

ax_rank[0].set_xlabel(f"Importance rank with {labels[modes_dict_rev[mode_base]]}", fontsize=plot_params["fs_ax_label"])
ax_rank[1].set_xlabel(f"Importance rank with {labels[modes_dict_rev[mode_base]]}", fontsize=plot_params["fs_ax_label"])
ax_rank[0].set_ylabel(f"Importance rank with {labels[modes_dict_rev[mode_comp]]}", fontsize=plot_params["fs_ax_label"])

ax_rank[0].set_title("W/o existing infrastructure", fontsize=plot_params["fs_ax_label"])
ax_rank[1].set_title("With existing infrastructure", fontsize=plot_params["fs_ax_label"])

fig_rank.savefig(join(paths["plot_folder"], save, f"{save}_rank_{mode_base}_{mode_comp}_street_type.{plot_params['plot_format']}"), bbox_inches="tight")
fig_rank.savefig(join(paths["plot_folder"], save, f"{save}_rank_{mode_base}_{mode_comp}_street_type.png"), bbox_inches="tight")

plt.show()

In [ ]:
fig_rank, ax_rank = plt.subplots(ncols=2, figsize=(6, 3), dpi=plot_params["dpi"], sharey=True, sharex=True)

mode_comp = "050"
mode_comp_exinf = f"{mode_comp[:2]}1"

cmap_030_050 = LinearSegmentedColormap.from_list("BaseDarkComp", ((0.000, colors[modes_dict_rev[mode_comp]]), (0.500, color_street_nw_both), (1.000, colors[modes_dict_rev[mode_base]])))
cmap_030_050.set_bad(color_street_nw)

ordering_unused = pd.DataFrame({
    "segment": ordering[ordering["segment"].isin(unused_segments[mode_base]) & ordering["segment"].isin(unused_segments[mode_comp])]["segment"],
    mode_base: ordering[ordering["segment"].isin(unused_segments[mode_base]) & ordering["segment"].isin(unused_segments[mode_comp])][mode_base],
    mode_comp: ordering[ordering["segment"].isin(unused_segments[mode_base]) & ordering["segment"].isin(unused_segments[mode_comp])][mode_comp],
})
ordering_unused["diff"] = ordering_unused[mode_comp] - ordering_unused[mode_base]

ordering_unused_exinf = pd.DataFrame({
    "segment": ordering[ordering["segment"].isin(unused_segments[mode_base_exinf]) & ordering["segment"].isin(unused_segments[mode_comp_exinf])]["segment"],
    mode_base_exinf: ordering[ordering["segment"].isin(unused_segments[mode_base_exinf]) & ordering["segment"].isin(unused_segments[mode_comp_exinf])][mode_base_exinf],
    mode_comp_exinf: ordering[ordering["segment"].isin(unused_segments[mode_base_exinf]) & ordering["segment"].isin(unused_segments[mode_comp_exinf])][mode_comp_exinf]
})
ordering_unused_exinf["diff"] = ordering_unused_exinf[mode_comp_exinf] - ordering_unused_exinf[mode_base_exinf]

ordering_used = ordering[~ordering["segment"].isin(ordering_unused["segment"])][["segment", mode_base, mode_comp]]
ordering_used["diff"] = ordering_used[mode_comp] - ordering_used[mode_base]

ordering_used_exinf = ordering[~ordering["segment"].isin(ordering_unused_exinf["segment"])][["segment", mode_base_exinf, mode_comp_exinf]]
ordering_used_exinf["diff"] = ordering_used_exinf[mode_comp_exinf] - ordering_used_exinf[mode_base_exinf]

segment_ordering_color = {seg: cmap_030_050(divnorm_ranking(ordering_diff[ordering["segment"] == seg][mode_comp].iloc[0])) for seg in ordering_used["segment"]}
segment_ordering_color = segment_ordering_color | {seg: color_street_nw for seg in ordering_unused["segment"]}
segment_ordering_color_exinf = {seg: cmap_030_050(divnorm_ranking(ordering_diff[ordering["segment"] == seg][mode_comp_exinf].iloc[0])) for seg in ordering_used_exinf["segment"]}
segment_ordering_color_exinf = segment_ordering_color_exinf | {seg: color_street_nw for seg in ordering_unused_exinf["segment"]}

ax_rank[0].plot(np.arange(0, 16000), np.arange(0, 16000), "--", c="#808080", alpha=0.75, lw=0.5, zorder=2)
ax_rank[0].plot(np.arange(0, 16000), np.arange(0, 16000)+outlier_threshold, "--", c="#808080", alpha=0.5, lw=0.5, zorder=2)
ax_rank[0].plot(np.arange(0, 16000), np.arange(0, 16000)-outlier_threshold, "--", c="#808080", alpha=0.5, lw=0.5, zorder=2)
ax_rank[0].scatter(ordering_unused[mode_base], ordering_unused[mode_comp], s=3, c=color_street_nw, alpha=0.5, linewidths=0, zorder=3)
rank_scatter = ax_rank[0].scatter(ordering_used[mode_base], ordering_used[mode_comp], s=5, c=ordering_used["diff"], norm=divnorm_ranking, cmap=cmap_030_050, linewidths=0, zorder=3)

ax_rank[1].plot(np.arange(0, 16000), np.arange(0, 16000), "--", c="#808080", alpha=0.75, lw=0.5, zorder=2)
ax_rank[1].plot(np.arange(0, 16000), np.arange(0, 16000)+outlier_threshold, "--", c="#808080", alpha=0.5, lw=0.5, zorder=2)
ax_rank[1].plot(np.arange(0, 16000), np.arange(0, 16000)-outlier_threshold, "--", c="#808080", alpha=0.5, lw=0.5, zorder=2)
ax_rank[1].scatter(ordering_unused_exinf[mode_base_exinf], ordering_unused_exinf[mode_comp_exinf], s=3, c=color_street_nw, alpha=0.5, linewidths=0, zorder=3)
rank_scatter_exinf = ax_rank[1].scatter(ordering_used_exinf[mode_base_exinf], ordering_used_exinf[mode_comp_exinf], s=5, c=ordering_used_exinf["diff"], norm=divnorm_ranking, cmap=cmap_030_050, linewidths=0, zorder=3)

ax_rank[0].set_xlim(0, 16000)
ax_rank[0].set_ylim(0, 16000)
ax_rank[0].set_aspect("equal")
ax_rank[1].set_aspect("equal")

ax_rank[0].tick_params(axis="both", labelsize=plot_params["fs_ax_ticks"])
ax_rank[1].tick_params(axis="both", labelsize=plot_params["fs_ax_ticks"])
ax_rank[0].ticklabel_format(axis="both", style="sci", scilimits=(-4, 4), useMathText=True)
ax_rank[1].ticklabel_format(axis="both", style="sci", scilimits=(-4, 4), useMathText=True)
ax_rank[0].xaxis.set_major_locator(ticker.MultipleLocator(2500))
ax_rank[0].yaxis.set_major_locator(ticker.MultipleLocator(2500))
ax_rank[0].xaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])
ax_rank[1].xaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])
ax_rank[0].yaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])
ax_rank[1].yaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])

ax_rank[0].set_xlabel(f"Importance rank in {labels[modes_dict_rev[mode_base]]}", fontsize=plot_params["fs_ax_label"])
ax_rank[1].set_xlabel(f"Importance rank in {labels[modes_dict_rev[mode_base]]}", fontsize=plot_params["fs_ax_label"])
ax_rank[0].set_ylabel(f"Importance rank in {labels[modes_dict_rev[mode_comp]]}", fontsize=plot_params["fs_ax_label"])

ax_rank[0].set_title("W/o existing infrastructure", fontsize=plot_params["fs_ax_label"])
ax_rank[1].set_title("With existing infrastructure", fontsize=plot_params["fs_ax_label"])

fig_rank.savefig(join(paths["plot_folder"], save, f"{save}_rank_{mode_base}_{mode_comp}.{plot_params['plot_format']}"), bbox_inches="tight")
plt.show()
cbformat = ticker.ScalarFormatter()
cbformat.set_scientific("{x:.2e}")
cbformat.set_powerlimits((-4, 4))
cbformat.set_useMathText(True)
cbar = fig_rank.colorbar(rank_scatter, ax=ax_rank[1], orientation="vertical", ticks=[-outlier_threshold, -int(outlier_threshold/2), 0, int(outlier_threshold/2), outlier_threshold], format="{x:.2e}")
cbar.ax.set_ylabel(r"Ranking difference $\Delta \mathrm{RANK}_{q_{\widetilde{\Delta T}},q_{\widetilde{\Delta T}, \mathrm{capped}}}$", fontsize=plot_params["fs_ax_label"])
cbar.ax.set_yticklabels([f"-{outlier_threshold}", f"-{int(outlier_threshold/2)}", "0", f"{int(outlier_threshold/2)}", f"{outlier_threshold}"])
cbar.ax.tick_params(labelsize=plot_params["fs_ax_ticks"])
fig_rank.savefig(join(paths["plot_folder"], save, f"{save}_rank_{mode_base}_{mode_comp}.png"), bbox_inches="tight")
#plt.show()
ax_rank[0].remove()
ax_rank[1].remove()
fig_rank.savefig(join(paths["plot_folder"], save, f"{save}_rank_{mode_base}_{mode_comp}_cbar.{plot_params['plot_format']}"), bbox_inches="tight")

In [ ]:
print(f"segments outside threshold w/o ex inf: {len(ordering_used[abs(ordering_used['diff']) > 120])/len(ordering_used)}")
print(f"segments outside threshold with ex inf: {len(ordering_used_exinf[abs(ordering_used_exinf['diff']) > 100])}")

print(f"segments outside threshold w/o ex inf: {len(ordering_used[abs(ordering_used['diff']) > outlier_threshold])}")
print(f"segments outside threshold with ex inf: {len(ordering_used_exinf[abs(ordering_used_exinf['diff']) > outlier_threshold])}")

In [ ]:
fig_rank_map_outliers, ax_rank_map_outliers = plt.subplots(ncols=2, figsize=plot_params["figsize_wide"], dpi=plot_params["dpi"],)

for plot_idx, ax in enumerate(fig_rank_map_outliers.axes):
    edge_color = list()
    edge_alpha = list()
    edge_zorder = list()
    edge_lw = list()
    if plot_idx == 0:
        for e in g.edges():
            if edge2segid[e] in ordering_unused["segment"]:
                edge_color.append(color_street_nw)
                edge_alpha.append(0.5)
                edge_zorder.append(1)
                edge_lw.append(0.2)
            else:
                e_diff = ordering_diff[ordering["segment"] == edge2segid[e]][mode_comp].iloc[0]
                e_color_i = cmap_030_050(divnorm_ranking(e_diff))
                e_alpha_i = 1
                e_zorder_i = 3
                e_lw_i = 0.5
                if abs(e_diff) <= outlier_threshold:
                    e_color_i = color_street_nw
                    e_alpha_i = 0.5
                    e_zorder_i = 1
                    e_lw_i = 0.2
                edge_color.append(e_color_i)
                edge_alpha.append(e_alpha_i)
                edge_zorder.append(e_zorder_i)
                edge_lw.append(e_lw_i)
    else:
        for e in g.edges():
            if edge2segid[e] in ordering_unused_exinf["segment"]:
                edge_color.append(color_street_nw)
                edge_alpha.append(0.5)
                edge_zorder.append(1)
                edge_lw.append(0.2)
            else:
                e_diff = ordering_diff[ordering["segment"] == edge2segid[e]][mode_comp].iloc[0]
                e_color_i = cmap_030_050(divnorm_ranking(e_diff))
                e_alpha_i = 1
                e_zorder_i = 3
                e_lw_i = 0.5
                if abs(e_diff) <= outlier_threshold:
                    e_color_i = color_street_nw
                    e_alpha_i = 0.5
                    e_zorder_i = 1
                    e_lw_i = 0.2
                edge_color.append(e_color_i)
                edge_alpha.append(e_alpha_i)
                edge_zorder.append(e_zorder_i)
                edge_lw.append(e_lw_i)

    plot_graph(g, ax=ax, node_size=0, edge_linewidth=edge_lw, edge_color=edge_color, edge_alpha=edge_alpha, edge_zorder=edge_zorder, show=False)

fig_rank_map_outliers.savefig(join(paths["plot_folder"], save, f"{save}_rank_{mode_base}_{mode_comp}_map_outliers.png"), bbox_inches="tight") #{plot_params['plot_format']}
plt.show()

In [ ]:
fig_rank_map, ax_rank_map = plt.subplots(ncols=2, figsize=plot_params["figsize_wide"], dpi=plot_params["dpi"],)

for plot_idx, ax in enumerate(fig_rank_map.axes):
    edge_color = list()
    edge_alpha = list()
    edge_zorder = list()
    edge_lw = list()
    if plot_idx == 0:
        for e in g.edges():
            if edge2segid[e] in ordering_unused["segment"]:
                edge_color.append(color_street_nw)
                edge_alpha.append(0.5)
                edge_zorder.append(1)
                edge_lw.append(0.2)
            else:
                e_diff = ordering_diff[ordering["segment"] == edge2segid[e]][mode_comp].iloc[0]
                edge_color.append(cmap_030_050(divnorm_ranking(e_diff)))
                e_alpha_i = 1
                e_zorder_i = 3
                e_lw_i = 0.5
                if abs(e_diff) <= outlier_threshold:
                    e_alpha_i = 0.5
                    e_zorder_i = 2
                    e_lw_i = 0.3
                edge_alpha.append(e_alpha_i)
                edge_zorder.append(e_zorder_i)
                edge_lw.append(e_lw_i)
    else:
        for u, v, d in g.edges(data="ex_inf"):
            e = (u, v)
            if d:
                edge_color.append(color_street_nw_exinf)
                edge_zorder.append(2)
                edge_lw.append(0.2)
            elif edge2segid[e] in ordering_unused_exinf["segment"]:
                edge_color.append(color_street_nw)
                edge_zorder.append(1)
                edge_lw.append(0.2)
            else:
                e_diff = ordering_diff[ordering["segment"] == edge2segid[e]][mode_comp].iloc[0]
                edge_color.append(cmap_030_050(divnorm_ranking(e_diff)))
                e_alpha_i = 1
                e_zorder_i = 4
                e_lw_i = 0.5
                if abs(e_diff) <= outlier_threshold:
                    e_alpha_i = 0.5
                    e_zorder_i = 3
                    e_lw_i = 0.3
                edge_alpha.append(e_alpha_i)
                edge_zorder.append(e_zorder_i)
                edge_lw.append(e_lw_i)

    plot_graph(g, ax=ax, node_size=0, edge_linewidth=edge_lw, edge_color=edge_color, edge_alpha=edge_alpha, edge_zorder=edge_zorder, show=False)

fig_rank_map.savefig(join(paths["plot_folder"], save, f"{save}_rank_{mode_base}_{mode_comp}_map.png"), bbox_inches="tight") #{plot_params['plot_format']}
plt.show()

In [ ]:
fig_core_struc, ax_core_struc = plt.subplots(figsize=(3.5, 3.5), dpi=plot_params["dpi"])
plot_point = ba_80_bpp[mode_base]
bbp_comp_idx = bpp.loc[(bpp[mode_base]-plot_point).abs().argsort()[:1]][mode_base].index.tolist()[0]
print(f"{bbp_comp_idx}: {bpp[mode_base].loc[bbp_comp_idx]:1.3f}")

removed_bike_paths = [segid2edge[s_id] for s_id in order_segment[mode_base].loc[bbp_comp_idx:]]

edge_width = [0.5 if e not in removed_bike_paths else 0.2 for e in g.edges()]
edge_color = [colors[modes_dict_rev[mode_base]] if e not in removed_bike_paths else color_street_nw for e in g.edges()]
edge_zorder = [3 if e not in removed_bike_paths else 2 for e in g.edges()]

plot_graph(g, ax=ax_core_struc, node_size=0, edge_linewidth=edge_width, edge_color=edge_color, edge_zorder=edge_zorder, show=False)

fig_core_struc.savefig(join(paths["plot_folder"], save, f"{save}_core_structure_{mode_base}.{plot_params['plot_format']}"), bbox_inches="tight")
plt.show()

In [ ]:
fig_core_struc_exinf, ax_core_struc_exinf = plt.subplots(figsize=(3.5, 3.5), dpi=plot_params["dpi"])
plot_point = ba_80_bpp_exinf[mode_base_exinf]
bbp_comp_idx = bpp.loc[(bpp[mode_base]-plot_point).abs().argsort()[:1]][mode_base].index.tolist()[0]
print(f"{bbp_comp_idx}: {bpp[mode_base].loc[bbp_comp_idx]:1.3f}")

removed_bike_paths = [segid2edge[s_id] for s_id in order_segment[mode_base].loc[bbp_comp_idx:]]

edge_width = list()
edge_color = list()
edge_zorder = list()
for u, v, d in g.edges(data="ex_inf"):
    if d:
        edge_width.append(0.5)
        edge_color.append(color_street_nw_exinf)
        edge_zorder.append(2)
    elif (u, v) not in removed_bike_paths:
        edge_width.append(0.5)
        edge_color.append(colors[modes_dict_rev[mode_base_exinf]])
        edge_zorder.append(3)
    else:
        edge_width.append(0.2)
        edge_color.append(color_street_nw)
        edge_zorder.append(1)

plot_graph(g, ax=ax_core_struc_exinf, node_size=0, edge_linewidth=edge_width, edge_color=edge_color, edge_zorder=edge_zorder, show=False)

fig_core_struc_exinf.savefig(join(paths["plot_folder"], save, f"{save}_core_structure_exinf_{mode_base_exinf}.{plot_params['plot_format']}"), bbox_inches="tight")
plt.show()

In [ ]:
sum([d["length"] for _, _, d in g.edges(data=True) if d["ex_inf"]]) / sum([d for _, _, d in g.edges(data="length")])

In [ ]:
sum([d["length"] for _, _, d in g.edges(data=True) if d["ex_inf"] and d["highway"] == "bike_path"]) / (1000*470)

In [ ]:
exinf_plot_base = "030"

fig_exinf_comp, ax_exinf_comp = plt.subplots(figsize=(3.5, 3.5), dpi=plot_params["dpi"])

plot_point = sum([d["length"] for _, _, d in g.edges(data=True) if d["ex_inf"] and not d["highway"] == "bike_path"]) / sum([d for _, _, d in g.edges(data="length")])
bbp_idx = bpp.loc[(bpp[exinf_plot_base]-plot_point).abs().argsort()[:1]][exinf_plot_base].index.tolist()[0]
print(f"Bikeability algo: {ba[exinf_plot_base].loc[bbp_idx]:1.3f}")
print(f"Bikeability exinf: {ba['001'].min():1.3f}")
removed_bike_paths_base = [segid2edge[s_id] for s_id in order_segment[exinf_plot_base].loc[bbp_idx:]]
bike_paths_exinf = [(u, v) for u, v, d in g.edges(data=True) if d["ex_inf"] and not d["highway"] == "bike_path"]

edge_width = list()
edge_color = list()
edge_zorder = list()
for edge in g.edges():
    if edge not in removed_bike_paths_base and edge in bike_paths_exinf:
        edge_width.append(0.5)
        edge_color.append("#469990")
        edge_zorder.append(5)
    elif edge not in removed_bike_paths_base and edge not in bike_paths_exinf:
        edge_width.append(0.5)
        edge_color.append(colors[modes_dict_rev[exinf_plot_base]])
        edge_zorder.append(4)
    elif edge in removed_bike_paths_base and edge in bike_paths_exinf:
        edge_width.append(0.5)
        edge_color.append(color_street_nw_exinf)
        edge_zorder.append(4)
    else:
        edge_width.append(0.2)
        edge_color.append(color_street_nw)
        edge_zorder.append(1)

plot_graph(g, ax=ax_exinf_comp, node_size=0, edge_linewidth=edge_width, edge_color=edge_color, edge_zorder=edge_zorder, show=False)
#ax_exinf_comp.set_title(f"Norm. len. of bike paths $\lambda = {plot_point:1.2f}$", fontsize=plot_params["fs_ax_label"], pad=2)

fig_exinf_comp.savefig(join(paths["plot_folder"], save, f"{save}_exinf_comp_{exinf_plot_base}.{plot_params['plot_format']}"), bbox_inches="tight")
plt.show()

## Noisy Data

In [ ]:
order_segment_noisy = pd.DataFrame()

bpp_noisy_single_runs = pd.DataFrame()
bpp_noisy = pd.DataFrame()
utility_noisy = pd.DataFrame()
ba_noisy_single_runs = pd.DataFrame()

for noisy in noise_params:
    for mode_noisy in modes_noisy:
        for i in range(1, noise_samples+1):
            save_noisy = f"{save}_{noisy}_{i}"

            with open(join(paths["output_folder"], save_noisy, f"{save_noisy}_order_{mode_noisy}_eval.json"), "r") as f:
                order_edges_mode_noisy = json.load(f)
            order_segment_noisy_i = pd.DataFrame({f"{noisy}_{mode_noisy}_{i}": [edge2segid[tuple(e)] for e in reversed(order_edges_mode_noisy)]})
            order_segment_noisy = pd.concat((order_segment_noisy, order_segment_noisy_i), axis=1)

            with open(join(paths["output_folder"], save_noisy, f"{save_noisy}_bpp_{mode_noisy}_eval.json"), "r") as f:
                bpp_mode_noisy = json.load(f)
            bpp_noisy_single_run_i = pd.DataFrame({f"{noisy}_{mode_noisy}_{i}": list(reversed(bpp_mode_noisy))})
            bpp_noisy_single_runs = pd.concat((bpp_noisy_single_runs, bpp_noisy_single_run_i), axis=1)

            with open(join(paths["output_folder"], save_noisy, f"{save_noisy}_utility_{mode_noisy}_eval.json"), "r") as f:
                utility_mode_noisy = json.load(f)
            utility_noisy_i = pd.DataFrame({f"{noisy}_{mode_noisy}_{i}": list(reversed(utility_mode_noisy))})
            utility_noisy = pd.concat((utility_noisy, utility_noisy_i.replace(to_replace=0.0, value=np.nan).dropna()), axis=1)

            ba_noisy_single_run_i = pd.DataFrame({f"{noisy}_{mode_noisy}_{i}": [(utility_base - i) / (utility_base - utility_opt) for i in utility_noisy[f"{noisy}_{mode_noisy}_{i}"]]})
            ba_noisy_single_runs = pd.concat((ba_noisy_single_runs, ba_noisy_single_run_i.replace(to_replace=0.0, value=np.nan).dropna()), axis=1)

        bpp_noisy[f"{noisy}_{mode_noisy}"] = bpp[mode_noisy]

ba_noisy = pd.DataFrame()

for noisy in noise_params:
    for mode_noisy in modes_noisy:
        ba_noisy_interpol = pd.DataFrame()
        for i in range(1, noise_samples+1):
            ba_noisy_interpol[i] = np.interp(bpp_noisy[f"{noisy}_{mode_noisy}"], bpp_noisy_single_runs[f"{noisy}_{mode_noisy}_{i}"], ba_noisy_single_runs[f"{noisy}_{mode_noisy}_{i}"])
        ba_noisy[f"{noisy}_{mode_noisy}"] = ba_noisy_interpol[range(1, noise_samples+1)].mean(axis=1)

In [ ]:
ordering_noisy = pd.DataFrame({"segment": sorted(order_segment["000"])})

for noisy in noise_params:
    for mode_noisy in modes_noisy:
        for i in range(1, noise_samples+1):
            ordering_noisy_i = pd.DataFrame({f"{noisy}_{mode_noisy}_{i}": ordering_noisy["segment"].map({seg: idx for idx, seg in enumerate(order_segment_noisy[f"{noisy}_{mode_noisy}_{i}"], start=1)})})
            ordering_noisy = pd.concat((ordering_noisy, ordering_noisy_i), axis=1)

In [ ]:
unused_segments_noisy = dict()
used_segments_noisy = dict()
for noisy in noise_params:
    for mode_noisy in modes_noisy:
        for i in range(1, noise_samples+1):
            mode_cut = ba_noisy_single_runs[f"{noisy}_{mode_noisy}_{i}"].loc[(ba_noisy_single_runs[f"{noisy}_{mode_noisy}_{i}"]>=1.0)].index[0]
            unused_segments_noisy[f"{noisy}_{mode_noisy}_{i}"] = order_segment_noisy[f"{noisy}_{mode_noisy}_{i}"].to_list()[mode_cut:]
            used_segments_noisy[f"{noisy}_{mode_noisy}_{i}"] = order_segment_noisy[f"{noisy}_{mode_noisy}_{i}"].to_list()[:mode_cut]

In [ ]:
fig_rob_ba, ax_rob_ba = plt.subplots(nrows=2, ncols=5, figsize=(6.5, 3.0), dpi=plot_params["dpi"], sharey=True, sharex=True)
fig_rob_ba.tight_layout(pad=1.0, h_pad=0.5, w_pad=0.1)

subplot_numbering = 0

for clm, noisy in enumerate(noise_params):
    ax_rob_ba[0, clm].hlines(0, xmin=0, xmax=1, lw=0.5, ls="--", color="#808080")
    ax_rob_ba[1, clm].hlines(0, xmin=0, xmax=1, lw=0.5, ls="--", color="#808080")
    #ax_rob_ba[0, clm].text(0.02, 0.91, chr(ord("a") + subplot_numbering), fontsize=8, fontweight="bold", transform=ax_rob_ba[0, clm].transAxes)
    #ax_rob_ba[1, clm].text(0.02, 0.91, chr(ord("a") + subplot_numbering + 5), fontsize=8, fontweight="bold", transform=ax_rob_ba[1, clm].transAxes)
    for mode_noisy in modes_noisy:
        row = int(mode_noisy[2])
        mode_noisy_label = modes_dict_rev[mode_noisy]
        ax_rob_ba[row, clm].plot(bpp_noisy[f"{noisy}_{mode_noisy}"], ba_noisy[f"{noisy}_{mode_noisy}"]-ba[mode_noisy], lw=0.5, ls=line_style[mode_noisy_label], color=colors[mode_noisy_label], label=labels[mode_noisy_label])


        ax_rob_ba[row, clm].tick_params(axis="y", labelsize=plot_params["fs_ax_ticks"], pad=1.5)
        ax_rob_ba[row, clm].tick_params(axis="x", labelsize=plot_params["fs_ax_ticks"], pad=1.5)


    ax_rob_ba[0, clm].set_title(noise_labels[noisy], fontsize=plot_params["fs_ax_label"])
    subplot_numbering += 1

ax_rob_ba[0, 0].set_ylabel("W/o existing infrastructure \n Change in bikeability", fontsize=plot_params["fs_ax_label"])
ax_rob_ba[1, 0].set_ylabel("With existing infrastructure \n Change in bikeability", fontsize=plot_params["fs_ax_label"])
ax_rob_ba[1, 0].set_xlim(0, 1)
ax_rob_ba[1, 0].xaxis.set_major_locator(ticker.MultipleLocator(0.2))
#ax_rob_ba[1, 0].set_ylim(-0.02, 0.02)

ax_rob_ba[1, 2].set_xlabel("Normalised length of bike paths $\lambda$", fontsize=plot_params["fs_ax_label"])

fig_rob_ba.savefig(join(paths["plot_folder"], save, f"{save}_rob_ba.{plot_params['plot_format']}"))
plt.show()

In [ ]:
counts, bin_edges, _ = plt.hist([d_i for _, _, d in g.edges(data="turn_penalty") for d_i in d.values()], bins=3)
plt.show()
plt.bar(bin_edges[:-1], counts / counts.sum(), width=np.diff(bin_edges), align='edge')
plt.hlines(0.075, xmin=0, xmax=3, color="#808080")
plt.show()

In [ ]:
counts, bin_edges, _ = plt.hist([d for n, d in g.nodes(data="intersection_size")], bins=3)
plt.show()
plt.bar(bin_edges[:-1], counts / counts.sum(), width=np.diff(bin_edges), align='edge')
plt.show()

In [ ]:
fig_rob_rank, ax_rob_rank = plt.subplots(2, 5, figsize=(6.5, 2.7), dpi=plot_params["dpi"], sharey=True, sharex=True)
fig_rob_rank.tight_layout(pad=1.0, h_pad=0.5, w_pad=0.1)

subplot_numbering = 0

corr_coeffs = pd.DataFrame()
corr_pvalues = pd.DataFrame()

for clm, noisy in enumerate(noise_params):
    for row, mode_noisy in enumerate(["030", "031"]):
        ax_rob_rank[row, clm].plot(ordering_noisy["segment"], ordering_noisy["segment"], "--", c="#808080", alpha=0.5, lw=0.5, zorder=2)
        ax_rob_rank[row, clm].plot(ordering_noisy["segment"], ordering_noisy["segment"]+outlier_threshold, "--", c="#808080", alpha=0.15, lw=0.5, zorder=2)
        ax_rob_rank[row, clm].plot(ordering_noisy["segment"], ordering_noisy["segment"]-outlier_threshold, "--", c="#808080", alpha=0.15, lw=0.5, zorder=2)
        #ax_rob_rank[row, clm].text(-0.1, 0.91, chr(ord("a") + subplot_numbering + 5 * row), fontsize=8, fontweight="bold", transform=ax_rob_rank[row, clm].transAxes)

        mode_noisy_label = modes_dict_rev[mode_noisy]

        rob_rank_x, rob_rank_y = [], []
        diff_order = []
        for i in range(1, noise_samples+1):
            ax_rob_rank[row, clm].scatter(ordering[mode_noisy], ordering_noisy[f"{noisy}_{mode_noisy}_{i}"], color=colors[mode_noisy_label], s=1, alpha=0.1, zorder=3)
            diff_order += (ordering[mode_noisy] - ordering_noisy[f"{noisy}_{mode_noisy}_{i}"]).abs().to_list()
            rob_rank_x += ordering[mode_noisy].to_list()
            rob_rank_y += ordering_noisy[f"{noisy}_{mode_noisy}_{i}"].to_list()
        corr_coeff_rank, corr_pvalues_rank = stats.kendalltau(rob_rank_x, rob_rank_y)
        corr_coeffs[mode_noisy] = [corr_coeff_rank]
        corr_pvalues[mode_noisy] = [corr_pvalues_rank]
        ax_rob_rank[row, clm].text(0.64, 0.05, fr"$\rho= {corr_coeff_rank:4.3f}$", fontsize=plot_params["fs_ax_ticks"], transform=ax_rob_rank[row, clm].transAxes)
        #ax_rob_rank[row, clm].text(0.30, 0.15, fr"avg. $\Delta RANK = {np.mean(diff_order):4.0f}$", fontsize=plot_params["fs_ax_ticks"], transform=ax_rob_rank[row, clm].transAxes)


        ax_rob_rank[row, clm].tick_params(axis="both", labelsize=plot_params["fs_ax_ticks"], pad=1.5)
        ax_rob_rank[row, clm].xaxis.set_major_locator(ticker.MultipleLocator(0.5e4))
        ax_rob_rank[row, clm].ticklabel_format(axis="both", style="sci", scilimits=(-4, 4), useMathText=True)
        ax_rob_rank[row, clm].xaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])
        ax_rob_rank[row, clm].yaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])
        ax_rob_rank[row, clm].set_aspect("equal")

    ax_rob_rank[0, clm].set_title(noise_labels[noisy], fontsize=plot_params["fs_ax_label"])
    subplot_numbering += 1

ax_rob_rank[1, 0].set_xlim(0, 16000)
ax_rob_rank[1, 0].set_ylim(0, 16000)
ax_rob_rank[1, 2].set_xlabel("Rank with base data", fontsize=plot_params["fs_ax_label"], labelpad=8.5)
ax_rob_rank[0, 0].set_ylabel(f"W/o existing bike paths \n Rank with noisy data", fontsize=plot_params["fs_ax_label"])
ax_rob_rank[1, 0].set_ylabel(f"With existing bike paths \n Rank with noisy data", fontsize=plot_params["fs_ax_label"])

for clm, noisy in enumerate(noise_params):
    for row, mode_noisy in enumerate(["030", "031"]):
        extent = ax_rob_rank[row, clm].get_window_extent().transformed(fig_rob_rank.dpi_scale_trans.inverted()).expanded(1.1, 1.2)
        #fig_rob_rank.savefig(join(paths["plot_folder"], save, f"{save}_rob_rank_{noisy}_{mode_noisy}.png"), bbox_inches=extent)
fig_rob_rank.savefig(join(paths["plot_folder"], save, f"{save}_rob_rank.png"))
plt.show()

## Appendix

In [ ]:
fig_rank, ax_rank = plt.subplots(ncols=2, figsize=(6, 3), dpi=plot_params["dpi"], sharey=True, sharex=True)

mode_comp = "020"
mode_comp_exinf = f"{mode_comp[:2]}1"

edge_lengths = [d for _, _, d in g.edges(data="length")]
divnorm_length = Normalize(vmin=0, vmax=max(edge_lengths), clip=True)
cmap = truncate_colormap(mpl.colormaps["Greys"], minval=0.2, maxval=1.0, n=256)
c_unused = [cmap(0.5)]

ordering_unused = pd.DataFrame({
    "segment": ordering[ordering["segment"].isin(unused_segments[mode_base]) & ordering["segment"].isin(unused_segments[mode_comp])]["segment"],
    mode_base: ordering[ordering["segment"].isin(unused_segments[mode_base]) & ordering["segment"].isin(unused_segments[mode_comp])][mode_base],
    mode_comp: ordering[ordering["segment"].isin(unused_segments[mode_base]) & ordering["segment"].isin(unused_segments[mode_comp])][mode_comp],
})
ordering_unused["length"] = [g.edges[segid2edge[seg]+(0,)]["length"] for seg in ordering_unused["segment"]]

ordering_unused_exinf = pd.DataFrame({
    "segment": ordering[ordering["segment"].isin(unused_segments[mode_base_exinf]) & ordering["segment"].isin(unused_segments[mode_comp_exinf])]["segment"],
    mode_base_exinf: ordering[ordering["segment"].isin(unused_segments[mode_base_exinf]) & ordering["segment"].isin(unused_segments[mode_comp_exinf])][mode_base_exinf],
    mode_comp_exinf: ordering[ordering["segment"].isin(unused_segments[mode_base_exinf]) & ordering["segment"].isin(unused_segments[mode_comp_exinf])][mode_comp_exinf]
})
ordering_unused_exinf["length"] = [g.edges[segid2edge[seg]+(0,)]["length"] for seg in ordering_unused_exinf["segment"]]

ordering_used = ordering[~ordering["segment"].isin(ordering_unused["segment"])][["segment", mode_base, mode_comp]]
ordering_used["length"] = [g.edges[segid2edge[seg]+(0,)]["length"] for seg in ordering_used["segment"]]

ordering_used_exinf = ordering[~ordering["segment"].isin(ordering_unused_exinf["segment"])][["segment", mode_base_exinf, mode_comp_exinf]]
ordering_used_exinf["length"] = [g.edges[segid2edge[seg]+(0,)]["length"] for seg in ordering_used_exinf["segment"]]

ax_rank[0].plot(np.arange(0, 16000), np.arange(0, 16000), "--", c="#808080", alpha=0.75, lw=0.5, zorder=2)
ax_rank[0].plot(np.arange(0, 16000), np.arange(0, 16000)+outlier_threshold, "--", c="#808080", alpha=0.5, lw=0.5, zorder=2)
ax_rank[0].plot(np.arange(0, 16000), np.arange(0, 16000)-outlier_threshold, "--", c="#808080", alpha=0.5, lw=0.5, zorder=2)
ax_rank[0].scatter(ordering_unused[mode_base], ordering_unused[mode_comp], s=1, c=ordering_unused["length"], norm=divnorm_length, cmap=cmap,  alpha=0.5, linewidths=0, zorder=3)
rank_scatter = ax_rank[0].scatter(ordering_used[mode_base], ordering_used[mode_comp], s=1, c=ordering_used["length"], norm=divnorm_length, cmap=cmap, linewidths=0, zorder=3)

ax_rank[1].plot(np.arange(0, 16000), np.arange(0, 16000), "--", c="#808080", alpha=0.75, lw=0.5, zorder=2)
ax_rank[1].plot(np.arange(0, 16000), np.arange(0, 16000)+outlier_threshold, "--", c="#808080", alpha=0.5, lw=0.5, zorder=2)
ax_rank[1].plot(np.arange(0, 16000), np.arange(0, 16000)-outlier_threshold, "--", c="#808080", alpha=0.5, lw=0.5, zorder=2)
ax_rank[1].scatter(ordering_unused_exinf[mode_base_exinf], ordering_unused_exinf[mode_comp_exinf], s=1, c=ordering_unused_exinf["length"], norm=divnorm_length, cmap=cmap,  alpha=0.5, linewidths=0, zorder=3)
rank_scatter_exinf = ax_rank[1].scatter(ordering_used_exinf[mode_base_exinf], ordering_used_exinf[mode_comp_exinf], s=1, c=ordering_used_exinf["length"], norm=divnorm_length, cmap=cmap, linewidths=0, zorder=3)

ax_rank[0].set_xlim(0, 16000)
ax_rank[0].set_ylim(0, 16000)
ax_rank[0].set_aspect("equal")
ax_rank[1].set_aspect("equal")

ax_rank[0].text(0.02, 0.91, "a", fontsize=8, fontweight="bold", transform=ax_rank[0].transAxes)
ax_rank[1].text(0.02, 0.91, "b", fontsize=8, fontweight="bold", transform=ax_rank[1].transAxes)

ax_rank[0].tick_params(axis="both", labelsize=plot_params["fs_ax_ticks"])
ax_rank[1].tick_params(axis="both", labelsize=plot_params["fs_ax_ticks"])
ax_rank[0].ticklabel_format(axis="both", style="sci", scilimits=(-4, 4), useMathText=True)
ax_rank[1].ticklabel_format(axis="both", style="sci", scilimits=(-4, 4), useMathText=True)
ax_rank[0].xaxis.set_major_locator(ticker.MultipleLocator(2500))
ax_rank[0].yaxis.set_major_locator(ticker.MultipleLocator(2500))
ax_rank[0].xaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])
ax_rank[1].xaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])
ax_rank[0].yaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])
ax_rank[1].yaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])

ax_rank[0].set_xlabel(f"Importance rank in {labels[modes_dict_rev[mode_base]]}", fontsize=plot_params["fs_ax_label"])
ax_rank[1].set_xlabel(f"Importance rank in {labels[modes_dict_rev[mode_base]]}", fontsize=plot_params["fs_ax_label"])
ax_rank[0].set_ylabel(f"Importance rank in {labels[modes_dict_rev[mode_comp]]}", fontsize=plot_params["fs_ax_label"])

ax_rank[0].set_title("W/o existing infrastructure", fontsize=plot_params["fs_ax_label"])
ax_rank[1].set_title("With existing infrastructure", fontsize=plot_params["fs_ax_label"])

fig_rank.savefig(join(paths["plot_folder"], save, f"{save}_rank_{mode_base}_{mode_comp}.png"), bbox_inches="tight")

#plt.show()

cbformat = ticker.ScalarFormatter()
cbformat.set_scientific("{x:.2e}")
cbformat.set_powerlimits((-4, 4))
cbformat.set_useMathText(True)
cbar = fig_rank.colorbar(rank_scatter, ax=ax_rank[1], orientation="vertical")
cbar.ax.set_ylabel(r"Segment length $l_s$ [m]", fontsize=plot_params["fs_ax_label"])
cbar.ax.tick_params(labelsize=plot_params["fs_ax_ticks"])
plt.show()
ax_rank[0].remove()
ax_rank[1].remove()
fig_rank.savefig(join(paths["plot_folder"], save, f"{save}_rank_{mode_base}_{mode_comp}_cbar.{plot_params['plot_format']}"), bbox_inches="tight")

In [ ]:
fig_rob_ba_all, ax_rob_ba_all = plt.subplots(4, 5, figsize=(6.5, 5.5), dpi=plot_params["dpi"], sharey=True, sharex=True)
fig_rob_ba_all.tight_layout(pad=1.0, h_pad=0.5, w_pad=0.1)

subplot_numbering = 0

for clm, noisy in enumerate(noise_params):
    for row, mode_noisy in enumerate([m for m in modes_noisy if m[2] != "1"]):
        mode_noisy_label = modes_dict_rev[mode_noisy]
        ax_rob_ba_all[row, clm].plot(bpp_noisy[f"{noisy}_{mode_noisy}"], ba_noisy[f"{noisy}_{mode_noisy}"]-ba[mode_noisy], lw=0.5, color=colors[mode_noisy_label], label=labels[mode_noisy_label])
        ax_rob_ba_all[row, clm].text(0.02, 0.91, chr(ord("a") + subplot_numbering + 5 * row), fontsize=8, fontweight="bold", transform=ax_rob_ba_all[row, clm].transAxes)

        for i in range(1, noise_samples+1):
            ax_rob_ba_all[row, clm].plot(bpp_noisy_single_runs[f"{noisy}_{mode_noisy}_{i}"], ba_noisy_single_runs[f"{noisy}_{mode_noisy}_{i}"]-ba[mode_noisy], lw=0.25, color=colors[mode_noisy_label], alpha=0.2)

        ax_rob_ba_all[row, clm].tick_params(axis="y", labelsize=plot_params["fs_ax_ticks"], pad=1.5)
        ax_rob_ba_all[row, clm].tick_params(axis="x", labelsize=plot_params["fs_ax_ticks"], pad=1.5)


        ax_rob_ba_all[row, 0].set_ylabel(f"{labels[mode_noisy_label]} \n Change in bikeability", fontsize=plot_params["fs_ax_label"])
    ax_rob_ba_all[0, clm].set_title(noise_labels[noisy], fontsize=plot_params["fs_ax_label"])
    subplot_numbering += 1

ax_rob_ba_all[3, 0].set_xlim(0, 1)
ax_rob_ba_all[3, 0].set_ylim(-0.05, 0.05)
ax_rob_ba_all[3, 0].xaxis.set_major_locator(ticker.MultipleLocator(0.2))
ax_rob_ba_all[3, 2].set_xlabel("Normalised length of bike paths $\lambda$", fontsize=plot_params["fs_ax_label"])

fig_rob_ba_all.savefig(join(paths["plot_folder"], save, f"{save}_rob_ba_all.{plot_params['plot_format']}"))
plt.show()

In [ ]:
fig_rob_rank, ax_rob_rank = plt.subplots(3, 5, figsize=(6.5, 4), dpi=plot_params["dpi"], sharey=True, sharex=True)
fig_rob_rank.tight_layout(pad=1.0, h_pad=0.5, w_pad=0.1)

subplot_numbering = 0

corr_coeffs = pd.DataFrame()
corr_pvalues = pd.DataFrame()

for clm, noisy in enumerate(noise_params):
    for row, mode_noisy in enumerate([m for m in modes_noisy if (m != mode_base) and (m[2] != "1")]):
        ax_rob_rank[row, clm].plot(ordering_noisy["segment"], ordering_noisy["segment"], "--", c="#808080", alpha=0.5, lw=0.5, zorder=2)
        ax_rob_rank[row, clm].plot(ordering_noisy["segment"], ordering_noisy["segment"]+outlier_threshold, "--", c="#808080", alpha=0.15, lw=0.5, zorder=2)
        ax_rob_rank[row, clm].plot(ordering_noisy["segment"], ordering_noisy["segment"]-outlier_threshold, "--", c="#808080", alpha=0.15, lw=0.5, zorder=2)
        ax_rob_rank[row, clm].text(0.02, 0.91, chr(ord("a") + subplot_numbering + 5 * row), fontsize=8, fontweight="bold", transform=ax_rob_rank[row, clm].transAxes)

        mode_noisy_label = modes_dict_rev[mode_noisy]

        rob_rank_x, rob_rank_y = [], []
        for i in range(1, noise_samples+1):
            ax_rob_rank[row, clm].scatter(ordering[mode_noisy], ordering_noisy[f"{noisy}_{mode_noisy}_{i}"], color=colors[mode_noisy_label], s=1, alpha=0.1, zorder=3)
            rob_rank_x += ordering[mode_noisy].to_list()
            rob_rank_y += ordering_noisy[f"{noisy}_{mode_noisy}_{i}"].to_list()
        corr_coeff_rank, corr_pvalues_rank = stats.kendalltau(rob_rank_x, rob_rank_y)
        corr_coeffs[mode_noisy] = [corr_coeff_rank]
        corr_pvalues[mode_noisy] = [corr_pvalues_rank]
        ax_rob_rank[row, clm].text(0.60, 0.05, fr"$\rho= {corr_coeff_rank:4.3f}$", fontsize=plot_params["fs_ax_ticks"], transform=ax_rob_rank[row, clm].transAxes)

        ax_rob_rank[row, clm].tick_params(axis="both", labelsize=plot_params["fs_ax_ticks"], pad=1.5)
        ax_rob_rank[row, clm].xaxis.set_major_locator(ticker.MultipleLocator(0.5e4))
        ax_rob_rank[row, clm].ticklabel_format(axis="both", style="sci", scilimits=(-4, 4), useMathText=True)
        ax_rob_rank[row, clm].xaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])
        ax_rob_rank[row, clm].yaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])
        ax_rob_rank[row, clm].set_aspect("equal")

        ax_rob_rank[row, 0].set_ylabel(f"{labels[mode_noisy_label]} \n Rank with noisy data", fontsize=plot_params["fs_ax_label"])
    ax_rob_rank[0, clm].set_title(noise_labels[noisy], fontsize=plot_params["fs_ax_label"])
    subplot_numbering += 1

ax_rob_rank[2, 0].set_xlim(0, 16000)
ax_rob_rank[2, 0].set_ylim(0, 16000)
ax_rob_rank[2, 2].set_xlabel("Rank with base data", fontsize=plot_params["fs_ax_label"], labelpad=8.5)

fig_rob_rank.savefig(join(paths["plot_folder"], save, f"{save}_rob_rank_all.png"))
plt.show()

In [ ]:
fig_rob_rank_exinf, ax_rob_rank_exinf = plt.subplots(3, 5, figsize=(6.5, 4.0), dpi=plot_params["dpi"], sharey=True, sharex=True)
fig_rob_rank_exinf.tight_layout(pad=1.0, h_pad=0.5, w_pad=0.1)

subplot_numbering = 0

corr_coeffs_exinf = pd.DataFrame()
corr_pvalues_exinf = pd.DataFrame()

for clm, noisy in enumerate(noise_params):
    for row, mode_noisy in enumerate([m for m in modes_noisy if (m != mode_base_exinf) and (m[2] == "1")]):
        ax_rob_rank_exinf[row, clm].plot(ordering_noisy["segment"], ordering_noisy["segment"], "--", c="#808080", alpha=0.5, lw=0.5, zorder=2)
        ax_rob_rank_exinf[row, clm].plot(ordering_noisy["segment"], ordering_noisy["segment"]+outlier_threshold, "--", c="#808080", alpha=0.15, lw=0.5, zorder=2)
        ax_rob_rank_exinf[row, clm].plot(ordering_noisy["segment"], ordering_noisy["segment"]-outlier_threshold, "--", c="#808080", alpha=0.15, lw=0.5, zorder=2)
        ax_rob_rank_exinf[row, clm].text(0.02, 0.91, chr(ord("a") + subplot_numbering + 5 * row), fontsize=8, fontweight="bold", transform=ax_rob_rank_exinf[row, clm].transAxes)

        mode_noisy_label = modes_dict_rev[mode_noisy]

        rob_rank_x, rob_rank_y = [], []
        for i in range(1, noise_samples+1):
            ax_rob_rank_exinf[row, clm].scatter(ordering[mode_noisy], ordering_noisy[f"{noisy}_{mode_noisy}_{i}"], color=colors[mode_noisy_label], s=1, alpha=0.1, zorder=3)
            rob_rank_x += ordering[mode_noisy].to_list()
            rob_rank_y += ordering_noisy[f"{noisy}_{mode_noisy}_{i}"].to_list()
        corr_coeff_rank, corr_pvalues_rank = stats.kendalltau(rob_rank_x, rob_rank_y)
        corr_coeffs_exinf[mode_noisy] = [corr_coeff_rank]
        corr_pvalues_exinf[mode_noisy] = [corr_pvalues_rank]
        ax_rob_rank_exinf[row, clm].text(0.60, 0.05, fr"$\rho= {corr_coeff_rank:4.3f}$", fontsize=plot_params["fs_ax_ticks"], transform=ax_rob_rank_exinf[row, clm].transAxes)

        ax_rob_rank_exinf[row, clm].tick_params(axis="both", labelsize=plot_params["fs_ax_ticks"], pad=1.5)
        ax_rob_rank_exinf[row, clm].xaxis.set_major_locator(ticker.MultipleLocator(0.5e4))
        ax_rob_rank_exinf[row, clm].ticklabel_format(axis="both", style="sci", scilimits=(-4, 4), useMathText=True)
        ax_rob_rank_exinf[row, clm].xaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])
        ax_rob_rank_exinf[row, clm].yaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])
        ax_rob_rank_exinf[row, clm].set_aspect("equal")

        ax_rob_rank_exinf[row, 0].set_ylabel(f"{labels[mode_noisy_label]} \n Rank with noisy data", fontsize=plot_params["fs_ax_label"])
    ax_rob_rank_exinf[0, clm].set_title(noise_labels[noisy], fontsize=plot_params["fs_ax_label"])
    subplot_numbering += 1

ax_rob_rank_exinf[2, 0].set_xlim(0, 16000)
ax_rob_rank_exinf[2, 0].set_ylim(0, 16000)
ax_rob_rank_exinf[2, 2].set_xlabel("Rank with base data", fontsize=plot_params["fs_ax_label"], labelpad=8.5)

fig_rob_rank_exinf.savefig(join(paths["plot_folder"], save, f"{save}_rob_rank_all_exinf.png"))
plt.show()

In [ ]:
rng = np.random.default_rng(seed=12345)

In [ ]:
fig, ax = plt.subplots(ncols=4, figsize=(6, 3), dpi=300, sharex=True, sharey=True)

for idx, mode_noisy in enumerate([m for m in modes_noisy if m[2] != "1"]):
    rob_rank_x, rob_rank_y = [], []
    for i in range(1, 100):
        rob_rank_x_i = list(range(len(used_segments[mode_noisy])+1, len(g.edges())+1))
        rng.shuffle(rob_rank_x_i)
        rob_rank_y_i = list(range(len(used_segments[mode_noisy])+1, len(g.edges())+1))
        rng.shuffle(rob_rank_y_i)
        rob_rank_x += list(range(1, len(used_segments[mode_noisy])+1)) + rob_rank_x_i
        rob_rank_y += list(range(1, len(used_segments[mode_noisy])+1)) + rob_rank_y_i
        ax[idx].scatter(list(range(1, len(used_segments[mode_noisy])+1)) + rob_rank_x_i, list(range(1, len(used_segments[mode_noisy])+1)) + rob_rank_y_i, color="C0", s=1, alpha=0.1, zorder=3)

    corr_coeff_rank, corr_pvalues_rank = stats.kendalltau(rob_rank_x, rob_rank_y)
    ax[idx].text(0.55, 0.05, fr"$\rho= {corr_coeff_rank:4.3f}$", fontsize=plot_params["fs_ax_ticks"], transform=ax[idx].transAxes)

    ax[idx].set_xlim(0, 16000)
    ax[idx].set_ylim(0, 16000)
    ax[idx].set_title(labels[modes_dict_rev[mode_noisy]], fontsize=plot_params["fs_ax_label"])
    ax[idx].tick_params(axis="both", labelsize=plot_params["fs_ax_ticks"], pad=1.5)
    ax[idx].xaxis.set_major_locator(ticker.MultipleLocator(0.5e4))
    ax[idx].ticklabel_format(axis="both", style="sci", scilimits=(-4, 4), useMathText=True)
    ax[idx].xaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])
    ax[idx].yaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])
    ax[idx].set_aspect("equal")
plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=4, figsize=(6,3), dpi=300, sharex=True, sharey=True)

for idx, mode_noisy in enumerate([m for m in modes_noisy if m[2] == "1"]):
    rob_rank_x, rob_rank_y = [], []
    for i in range(1, 100):
        rob_rank_x_i = list(range(len(used_segments[mode_noisy])+1, len(g.edges())+1))
        rng.shuffle(rob_rank_x_i)
        rob_rank_y_i = list(range(len(used_segments[mode_noisy])+1, len(g.edges())+1))
        rng.shuffle(rob_rank_y_i)
        rob_rank_x += list(range(1, len(used_segments[mode_noisy])+1)) + rob_rank_x_i
        rob_rank_y += list(range(1, len(used_segments[mode_noisy])+1)) + rob_rank_y_i
        ax[idx].scatter(list(range(1, len(used_segments[mode_noisy])+1)) + rob_rank_x_i, list(range(1, len(used_segments[mode_noisy])+1)) + rob_rank_y_i, color="C0", s=1, alpha=0.1, zorder=3)

    corr_coeff_rank, corr_pvalues_rank = stats.kendalltau(rob_rank_x, rob_rank_y)
    ax[idx].text(0.60, 0.05, fr"$\rho= {corr_coeff_rank:4.3f}$", fontsize=plot_params["fs_ax_ticks"], transform=ax[idx].transAxes)

    ax[idx].set_xlim(0, 16000)
    ax[idx].set_ylim(0, 16000)
    ax[idx].set_title(labels[modes_dict_rev[mode_noisy]], fontsize=plot_params["fs_ax_label"])
    ax[idx].tick_params(axis="both", labelsize=plot_params["fs_ax_ticks"], pad=1.5)
    ax[idx].xaxis.set_major_locator(ticker.MultipleLocator(0.5e4))
    ax[idx].ticklabel_format(axis="both", style="sci", scilimits=(-4, 4), useMathText=True)
    ax[idx].xaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])
    ax[idx].yaxis.offsetText.set_fontsize(plot_params["fs_ax_ticks"])
    ax[idx].set_aspect("equal")
plt.show()

In [ ]:
evo_plot = "030"

fig_evo, ax_evo = plt.subplots(nrows=1, ncols=3, figsize=(6.5, 3.5), dpi=plot_params["dpi"])
#plot_points = [0.8, 0.5, 0.4, 0.3, 0.2, 0.15, 0.1, 0.05, 0.01]
plot_points = [ba_80_bpp[evo_plot], ba_100_bpp[evo_plot]/2, ba_100_bpp[evo_plot]]

for plot_idx, ax in enumerate(fig_evo.axes):
    plot_point = plot_points[plot_idx]
    bbp_comp_idx = bpp.loc[(bpp[evo_plot]-plot_point).abs().argsort()[:1]][evo_plot].index.tolist()[0]
    print(f"{bbp_comp_idx}: {bpp[evo_plot].loc[bbp_comp_idx]:1.3f}")

    removed_bike_paths = [segid2edge[s_id] for s_id in order_segment[evo_plot].loc[bbp_comp_idx:]]

    edge_width = [0.25 if e not in removed_bike_paths else 0.2 for e in g.edges()]
    edge_color = [colors[modes_dict_rev[evo_plot]] if e not in removed_bike_paths else color_street_nw for e in g.edges()]
    edge_zorder = [3 if e not in removed_bike_paths else 2 for e in g.edges()]

    plot_graph(g, ax=ax, node_size=0, edge_linewidth=edge_width, edge_color=edge_color, edge_zorder=edge_zorder, show=False)
    ax.set_title(f"Norm. len. of bike paths $\lambda = {bpp[evo_plot].loc[bbp_comp_idx]:1.2f}$", fontsize=plot_params["fs_ax_label"], pad=2)

fig_evo.savefig(join(paths["plot_folder"], save, f"{save}_evo_{evo_plot}.png"), bbox_inches="tight")
plt.show()

In [ ]:
evo_plot_exinf = "031"

fig_evo_exinf, ax_evo_exinf = plt.subplots(nrows=1, ncols=3, figsize=(6.5, 3.5), dpi=plot_params["dpi"])
plot_points_exinf = [ba_80_bpp_exinf[evo_plot_exinf], ba_100_bpp_exinf[evo_plot_exinf]/2, ba_100_bpp_exinf[evo_plot_exinf]]

for plot_idx, ax in enumerate(fig_evo_exinf.axes):
    plot_point_exinf = plot_points_exinf[plot_idx]
    bbp_evo_idx = bpp.loc[(bpp[evo_plot_exinf]-plot_point_exinf).abs().argsort()[:1]][evo_plot_exinf].index.tolist()[0]
    print(f"{bbp_evo_idx}: {bpp[evo_plot_exinf].loc[bbp_evo_idx]:1.3f}")
    removed_bike_paths = [segid2edge[s_id] for s_id in order_segment[evo_plot_exinf].loc[bbp_evo_idx:]]

    edge_width = list()
    edge_color = list()
    edge_zorder = list()
    for u, v, d in g.edges(data="ex_inf"):
        if d:
            edge_width.append(0.3)
            edge_color.append(color_street_nw_exinf)
            edge_zorder.append(2)
        elif (u, v) not in removed_bike_paths:
            edge_width.append(0.3)
            edge_color.append(colors[modes_dict_rev[evo_plot_exinf]])
            edge_zorder.append(3)
        else:
            edge_width.append(0.2)
            edge_color.append(color_street_nw)
            edge_zorder.append(1)

    plot_graph(g, ax=ax, node_size=0, edge_linewidth=edge_width, edge_color=edge_color, edge_zorder=edge_zorder, show=False)
    ax.set_title(f"Norm. len. of bike paths $\lambda = {bpp[evo_plot_exinf].loc[bbp_evo_idx]:1.2f}$", fontsize=plot_params["fs_ax_label"], pad=2)

fig_evo_exinf.savefig(join(paths["plot_folder"], save, f"{save}_evo_{evo_plot_exinf}.png"), bbox_inches="tight")
plt.show()